In [ ]:

=========================================================================================================================*/
const COL_TIMESTAMP             = 0;  // Column 1
const COL_EMAIL_ADDRESS_FACULTY = 1;  // Column 2 (Assessor Email from the form)
const COL_RESIDENT_ROLL_NUMBER  = 2;  // Column 3 (Resident Roll Number from the form - USED FOR RESIDENT LOOKUP)
const COL_PG_PROGRAM            = 3;  // Column 4
const COL_YEAR                  = 4;  // Column 5
const COL_DATE                  = 5;  // Column 6
const COL_SPECIALITY            = 6;  // Column 7 (Specialty from the form - USED FOR PD LOOKUP)
const COL_EPA_YES_NO            = 7;  // Column 8
const COL_TOPIC                 = 8;  // Column 9
const COL_EPA_NUMBER            = 9;  // Column 10
const COL_CASE_DESCRIPTION      = 10; // Column 11
const COL_HISTORY               = 11; // Column 12 (Rating 1)
const COL_CLINICAL              = 12; // Column 13 (Rating 2)
const COL_INVESTIGATION         = 13; // Column 14 (Rating 3)
const COL_TREATMENT             = 14; // Column 15 (Rating 4)
const COL_FOLLOWUP              = 15; // Column 16 (Rating 5)
const COL_PROFESSIONALISM       = 16; // Column 17 (Rating 6)
const COL_OVERALL_CARE          = 17; // Column 18 (Rating 7)
const COL_AREAS_DONE_WELL       = 18; // Column 19 (Text comment 1 - USED FOR NARRATIVE SKILL calculation input)
const COL_ACTION_NEEDED         = 19; // Column 20 (Text comment 2 - USED FOR NARRATIVE SKILL calculation input)
const COL_ASSESSOR_NAME         = 20; // Column 21 (Assessing Doctor's Full Name from form) - ADDED
const COL_ASSESSOR_POSITION     = 21; // Column 22 (Assessor Position from form) - ADJUSTED INDEX

/* === Analytics columns start at 24 (X) ===========================================================================
   These are columns calculated by the script and written back to the sheet.
   The `writeAnalytics` function assumes these are sequential starting from COL_FULL_MARK.
=========================================================================================================================*/
const COL_FULL_MARK             = 23; // Column 24 (Calculated Possible Mark based on items rated) - ADDED (Used by writeAnalytics)
const COL_TOTAL_MARK            = 24; // Column 25 (Calculated Obtained Mark - Sum of Ratings) - NOTE: Name mismatch with analytics variable 'total'
const COL_PERCENTAGE            = 25; // Column 26 (Calculated Percentage)
const COL_MEAN                  = 26; // Column 27 (Calculated Mean of valid ratings)
const COL_SD                    = 27; // Column 28 (Calculated Standard Deviation of valid ratings)
const COL_VARIANCE              = 28; // Column 29 (Calculated Variance of valid ratings)
const COL_DISCRIMINATION        = 29; // Column 30 (Calculated Discrimination Category based on Variance)
const COL_NARRATIVE_SKILL       = 30; // Column 31 (Calculated Narrative Skill Category based on Action Needed word count)


/* === MAIN ON-SUBMIT HANDLER ========================================== */
function CbD_onSubmit(e) {
    Logger.log('CbD_onSubmit triggered');
    // Check for required constants - helpful during development if constants are missing
    if (typeof COL_ASSESSOR_NAME === 'undefined' || typeof COL_FULL_MARK === 'undefined') {
        Logger.log('Error: Required COLUMN MAP constants are not defined.');
        // Optionally send an error email to the script owner here
        return;
    }

    try {
        // Use the event object to get the sheet and range if available
        let ss, sheet, row, values;

        if (e && e.range) {
            Logger.log('Event object found with range');
            ss = e.source;
            sheet = e.range.getSheet();
            row = e.range.getRow();

            // Only process if this is the CbD sheet
            if (sheet.getName() !== 'CbD') {
                Logger.log('Submission not on CbD sheet: ' + sheet.getName());
                return;
            }
        } else {
            Logger.log('No event object or range found, using fallback');
            // Fallback if e is not provided or doesn't contain range
            ss = SpreadsheetApp.getActiveSpreadsheet();
            sheet = ss.getSheetByName('CbD');
            if (!sheet) {
                Logger.log('CbD sheet not found');
                // Optionally send an error email to the script owner here
                return;
            }
            row = sheet.getLastRow();
        }

        Logger.log(`Processing row ${row} on sheet ${sheet.getName()}`);

        // Check if row is valid (must be greater than 1 to avoid header)
        if (row <= 1) {
            Logger.log('Skipping header row or invalid row number: ' + row);
            return;
        }

        // Get all relevant form response values from the row
        try {
            // Get columns from Timestamp (1) up to Assessor Position (22).
            // This corresponds to indices 0 through COL_ASSESSOR_POSITION (21).
            // Adding 1 to the index gives the number of columns to retrieve.
            const numFormResponseCols = COL_ASSESSOR_POSITION + 1;
            values = sheet.getRange(row, 1, 1, numFormResponseCols).getValues()[0];
            Logger.log(`Retrieved ${values.length} columns of data from row ${row}`);

            // Basic check to ensure we got some data and it's not just an empty row below headers
            // Check if we got at least up to the Assessor Position column
            if (!values || values.length < numFormResponseCols || !values[COL_TIMESTAMP]) {
                Logger.log('Row data seems incomplete or invalid. Skipping processing.');
                return;
            }

        } catch (err) {
            Logger.log('Error getting values from row: ' + err.toString());
            // Log stack trace if available
            if (err.stack) {
                Logger.log('Stack trace: ' + err.stack);
            }
            // Optionally send an error email to the script owner here
            return;
        }

        // Log key values for debugging using the new constants
        if (values.length > COL_RESIDENT_ROLL_NUMBER) {
            Logger.log(`Resident Roll Number: ${values[COL_RESIDENT_ROLL_NUMBER]}`);
        }
        if (values.length > COL_SPECIALITY) { // Use COL_SPECIALITY
            Logger.log(`Specialty: ${values[COL_SPECIALITY]}`);
        }
        if (values.length > COL_EMAIL_ADDRESS_FACULTY) { // Use COL_EMAIL_ADDRESS_FACULTY
            Logger.log(`Assessor Email (from form): ${values[COL_EMAIL_ADDRESS_FACULTY]}`);
        }
        if (values.length > COL_ASSESSOR_NAME) { // Use COL_ASSESSOR_NAME
             Logger.log(`Assessor Name (from form): ${values[COL_ASSESSOR_NAME]}`);
        }


        // Process the submission
        const analytics = calculateAnalytics(values); // calculateAnalytics now uses DOPS logic for Discrimination/Narrative
        Logger.log('Analytics calculated: ' + JSON.stringify(analytics));

        writeAnalytics(sheet, row, analytics);

        // Wait a moment to ensure analytics are written before sending email
        Utilities.sleep(500);

        // Perform lookups needed for emails
        const rollNumber = values.length > COL_RESIDENT_ROLL_NUMBER ? values[COL_RESIDENT_ROLL_NUMBER] : '';
        const specialty = values.length > COL_SPECIALITY ? values[COL_SPECIALITY] : '';

        let studentObj = getStudentByRollNumber(rollNumber);
        let pdObj = getProgramDirector(specialty);

        // Send separate emails
        sendEmailResident(values, analytics, studentObj);
        sendEmailFacultyPD(values, analytics, studentObj, pdObj);


        Logger.log('CbD_onSubmit completed successfully for row ' + row);
    } catch (error) {
        Logger.log('Error in CbD_onSubmit: ' + error.toString());
        if (error.stack) {
            Logger.log('Stack trace: ' + error.stack);
        }
        // Optionally send an error email to the script owner here
    }
}

/* === ANALYTICS (MODIFIED FOR DOPS DISCRIMINATION/NARRATIVE LOGIC) ================================= */
function calculateAnalytics(v) {
    try {
        // Map ratings and convert to numbers, handling "U/C" as 0
        // Uses constants: COL_HISTORY, COL_CLINICAL, COL_INVESTIGATION, COL_TREATMENT, COL_FOLLOWUP, COL_PROFESSIONALISM, COL_OVERALL_CARE
        const ratings = [
            parseRating(v[COL_HISTORY]),
            parseRating(v[COL_CLINICAL]),
            parseRating(v[COL_INVESTIGATION]),
            parseRating(v[COL_TREATMENT]),
            parseRating(v[COL_FOLLOWUP]),
            parseRating(v[COL_PROFESSIONALISM]),
            parseRating(v[COL_OVERALL_CARE])
        ];

        // Count non-zero values to calculate observed items
        const observed = ratings.filter(n => n > 0).length;
        const fullMark = observed * 9; // 9 is max score per rating (assuming 0-9 scale)
        const total = ratings.reduce((s, n) => s + n, 0);
        const pct = fullMark > 0 ? (total / fullMark) * 100 : 0;

        // Calculate mean, excluding 0 values if any observation was made
        const validRatings = ratings.filter(n => n > 0);
        const mean = validRatings.length > 0 ?
            validRatings.reduce((s, n) => s + n, 0) / validRatings.length : 0;

        // Calculate variance and standard deviation (Using standard sample variance if > 1 rating)
        let variance = 0;
        let sd = 0;

        if (validRatings.length > 0) {
            // Sum of squared differences
            const sumSquaredDiff = validRatings.reduce((sum, val) => sum + Math.pow(val - mean, 2), 0);

            // If we have more than one rating, use sample variance (n-1 denominator)
            // otherwise use population variance (n denominator)
            const divisor = validRatings.length > 1 ? validRatings.length - 1 : validRatings.length;

            variance = divisor > 0 ? sumSquaredDiff / divisor : 0;
            sd = Math.sqrt(variance);
        }

        // Determine discrimination level based on DOPS logic (Variance)
        // Implementing as described: Variance = 0 -> Poor, Variance > 0 and < 0.5 -> Good, Variance >= 0.5 -> Moderate
        let discrimination;
        if (variance === 0) {
            discrimination = 'Poor'; // As per DOPS description
        } else if (variance > 0 && variance < 0.5) { // Explicitly check variance > 0
            discrimination = 'Good'; // As per DOPS description
        } else if (variance >= 0.5) {
            discrimination = 'Moderate'; // As per DOPS description
        } else {
            discrimination = 'N/A'; // Fallback for non-numeric or unexpected variance
        }


        // Evaluate narrative skill based on DOPS logic (word count of 'Action needed' after filtering)
        // Uses constant: COL_ACTION_NEEDED
        const actionNeededText = String(v[COL_ACTION_NEEDED] || '');
        const filteredWords = filterWords(actionNeededText); // Use helper function
        const lenAreaImp = filteredWords.length; // Length after filtering

        let narrativeSkill;
        if (lenAreaImp <= 5) {
            narrativeSkill = 'Needs Improvement'; // As per DOPS description
        } else if (lenAreaImp > 5 && lenAreaImp <= 10) {
            narrativeSkill = 'Can be better'; // As per DOPS description
        } else if (lenAreaImp > 10) {
            narrativeSkill = 'Is Great'; // As per DOPS description
        } else {
            narrativeSkill = 'N/A'; // Fallback
        }


        // Also calculate LenAreaWell based on DOPS description for completeness, although not used in email body
        const areasDoneWellText = String(v[COL_AREAS_DONE_WELL] || '');
        const filteredWellWords = filterWords(areasDoneWellText);
        const lenAreaWell = filteredWellWords.length;


        return {
            fullMark,       // Calculated Full Mark
            total,          // Calculated Obtained Mark
            pct,            // Calculated Percentage
            mean,           // Calculated Mean
            sd,             // Calculated Standard Deviation
            variance,       // Calculated Variance
            discrimination, // Calculated Discrimination Category
            narrativeSkill, // Calculated Narrative Skill Category
            lenAreaWell,    // Filtered word count for 'Areas Done Well'
            lenAreaImp      // Filtered word count for 'Action Needed'
        };
    } catch (error) {
        Logger.log('Error in calculateAnalytics: ' + error.toString());
        // Return default values in case of error
        return {
            fullMark: 0,
            total: 0,
            pct: 0,
            mean: 0,
            sd: 0,
            variance: 0,
            discrimination: 'N/A',
            narrativeSkill: 'N/A',
            lenAreaWell: 0,
            lenAreaImp: 0
        };
    }
}

// Helper function to parse rating values ( expecting 0-9 or 'U/C' )
function parseRating(value) {
    if (value === undefined || value === null || String(value).trim() === '') return 0;
    const stringValue = String(value).trim().toUpperCase();
    if (stringValue === 'U/C') return 0;
    const num = Number(value);
    // Treat non-numeric or numbers outside 0-9 as 0
    if (isNaN(num) || num < 0 || num > 9) {
         Logger.log(`Warning: Invalid rating value "${value}" encountered. Treating as 0.`);
         return 0;
    }
    return num;
}

// Helper function to filter words based on DOPS description (remove punctuation, split, filter words < 4 chars)
function filterWords(text) {
    if (!text) return [];
    // Remove common punctuation and split into words
    const words = String(text).toLowerCase().replace(/[.,!?;:"'()]/g, '').split(/\s+/);
    // Filter out empty strings and words less than 4 characters
    return words.filter(word => word.length >= 4);
}


// Writes the calculated analytics back to the sheet row
function writeAnalytics(sh, row, a) {
    try {
        // Uses constants for analytics columns: COL_FULL_MARK, COL_TOTAL_MARK, COL_PERCENTAGE, etc.
        // Writes the analytics object values to the sheet row starting at COL_FULL_MARK.
        // The order in the array must match the sequential order of the analytics columns in the sheet.
        sh.getRange(row, COL_FULL_MARK + 1, 1, 8).setValues([[ // +1 because getRange is 1-based, constants are 0-based index
            a.fullMark,       // Goes to Column COL_FULL_MARK (Index 23, Column 24)
            a.total,          // Goes to Column COL_TOTAL_MARK (Index 24, Column 25)
            a.pct,            // Goes to Column COL_PERCENTAGE (Index 25, Column 26)
            a.mean,           // Goes to Column COL_MEAN (Index 26, Column 27)
            a.sd,             // Goes to Column COL_SD (Index 27, Column 28)
            a.variance,       // Goes to Column COL_VARIANCE (Index 28, Column 29)
            a.discrimination, // Goes to Column COL_DISCRIMINATION (Index 29, Column 30)
            a.narrativeSkill  // Goes to Column COL_NARRATIVE_SKILL (Index 30, Column 31)
        ]]);
        Logger.log('Analytics written successfully to row ' + row);
    } catch (error) {
        Logger.log('Error in writeAnalytics: ' + error.toString());
        if (error.stack) {
            Logger.log('Stack trace: ' + error.stack);
        }
        // Optionally send an error email to the script owner here
    }
}

/* === EMAIL DISPATCH (SPLIT INTO TWO EMAILS - CbD CONTENT) =================== */

// Sends email to the Resident (using improved CbD format)
function sendEmailResident(rowVals, analytics, studentObj) {
    try {
        const residentName = studentObj && studentObj.name ? studentObj.name.trim() : 'Resident';
        const studentEmail = studentObj && studentObj.email ? studentObj.email.trim() : '';

        if (!studentEmail) {
            Logger.log('Resident email not found for sending email to resident. Skipping.');
            return;
        }

        // Safely access data from rowVals using constants (0-based index)
        const specialty = rowVals.length > COL_SPECIALITY ? rowVals[COL_SPECIALITY] : 'N/A';
        const dateValue = rowVals.length > COL_DATE ? rowVals[COL_DATE] : 'N/A';
        let dateStr = 'N/A';
        try {
            if (dateValue instanceof Date) {
                dateStr = dateValue.toLocaleDateString();
            } else {
                const parsedDate = new Date(dateValue);
                if (!isNaN(parsedDate.getTime())) {
                    dateStr = parsedDate.toLocaleDateString();
                } else {
                    dateStr = String(dateValue);
                }
            }
        } catch (e) { dateStr = String(dateValue); } // Safely get date

        const areasDoneWell = rowVals.length > COL_AREAS_DONE_WELL ? (String(rowVals[COL_AREAS_DONE_WELL] || '')).trim() || 'Not specified' : 'Not specified';
        const actionNeeded = rowVals.length > COL_ACTION_NEEDED ? (String(rowVals[COL_ACTION_NEEDED] || '')).trim() || 'Not specified' : 'Not specified';

         // Access resident details from lookup or form
        const residentRollNumber = studentObj && studentObj.rollNumber ? String(studentObj.rollNumber).trim() : (rowVals.length > COL_RESIDENT_ROLL_NUMBER ? rowVals[COL_RESIDENT_ROLL_NUMBER] : 'N/A');
        const year = rowVals.length > COL_YEAR ? rowVals[COL_YEAR] : 'N/A';
        const topic = rowVals.length > COL_TOPIC ? rowVals[COL_TOPIC] : 'N/A';
        const epaNumber = rowVals.length > COL_EPA_NUMBER ? rowVals[COL_EPA_NUMBER] : 'N/A';

        // Get Assessor Email from form for the introductory sentence
        const assessorEmailFromForm = rowVals.length > COL_EMAIL_ADDRESS_FACULTY ? String(rowVals[COL_EMAIL_ADDRESS_FACULTY] || '').trim() : 'Assessor';


        // Access individual rating scores using constants (0-based index)
        const historyRating = rowVals.length > COL_HISTORY ? rowVals[COL_HISTORY] : 'N/A';
        const clinicalRating = rowVals.length > COL_CLINICAL ? rowVals[COL_CLINICAL] : 'N/A';
        const investigationRating = rowVals.length > COL_INVESTIGATION ? rowVals[COL_INVESTIGATION] : 'N/A';
        const treatmentRating = rowVals.length > COL_TREATMENT ? rowVals[COL_TREATMENT] : 'N/A';
        const followupRating = rowVals.length > COL_FOLLOWUP ? rowVals[COL_FOLLOWUP] : 'N/A';
        const professionalismRating = rowVals.length > COL_PROFESSIONALISM ? rowVals[COL_PROFESSIONALISM] : 'N/A';
        const overallCareRating = rowVals.length > COL_OVERALL_CARE ? rowVals[COL_OVERALL_CARE] : 'N/A';


        const analyticsData = analytics || { fullMark: 0, total: 0, pct: 0, mean: 0, sd: 0, variance: 0, discrimination: 'N/A', narrativeSkill: 'N/A' };
        const obtainedMarks = analyticsData.total || 0;
        const obtainedPercentage = (analyticsData.pct || 0).toFixed(2);


        // Construct the email body based on the latest draft for the resident
        const subject = `CbD Feedback – ${residentName} (${specialty}) – ${dateStr}`; // Added date to subject
        const body = `Dear ${residentName},\n\n` +
            `Please find below your evaluation results from your recent Case-based Discussion (CbD) assessment completed on ${dateStr} with ${assessorEmailFromForm}.\n\n` + // Use assessor email here
            `This CbD assessment is designed to provide structured feedback on your clinical reasoning and patient management skills, contributing to your ongoing development towards competency.\n\n` +

            `--- Resident Details ---\n` +
            `Resident Name: ${residentName}\n` +
            `Resident Roll Number: ${residentRollNumber}\n` +
            `Year: ${year}\n` +
            `Speciality: ${specialty}\n` +
            `EPA Topic: ${topic}\n` +
            `EPA Number: ${epaNumber}\n\n` +

            `Student Evaluation:\n\n` + // Header for ratings
            `History Taking: ${historyRating}\n` +
            `Clinical Assessment: ${clinicalRating}\n` +
            `Investigation and Referral: ${investigationRating}\n` +
            `Treatment: ${treatmentRating}\n` +
            `Follow-up and Future Planning: ${followupRating}\n` +
            `Professionalism: ${professionalismRating}\n` +
            `Overall Clinical Care: ${overallCareRating}\n\n` +

            `Obtained Marks: ${obtainedMarks}\n` + // Include obtained marks
            `Obtained Percentage: ${obtainedPercentage}%\n\n` + // Include obtained percentage

            `--- Feedback from Assessor ---\n\n` + // Section header
            `Areas done well:\n${areasDoneWell}\n\n` + // Include comments
            `Specific action needed to achieve competency:\n${actionNeeded}\n\n` + // Include comments

            `---\n\n` + // Separator

            `We encourage you to review this feedback and discuss it with your Program Director or mentor to support your ongoing professional growth.\n\n` + // Encouragement

            `Thank you\n` + // Added "Thank you"
            `Dean Office`; // Added "Dean Office"


        MailApp.sendEmail({
            to: studentEmail,
            subject: subject,
            body: body
        });

        Logger.log('Email sent to Resident: ' + studentEmail);

    } catch (err) {
        Logger.log('Error sending email to Resident: ' + err.toString());
        if (err.stack) {
            Logger.log('Stack trace: ' + err.stack);
        }
        // Optionally send an error email to the script owner here
    }
}

// Sends email to Faculty and Program Director (concise notification)
function sendEmailFacultyPD(rowVals, analytics, studentObj, pdObj) {
    try {
        // Access data using variables and constants
        const programDirectorName = pdObj && pdObj.name ? pdObj.name.trim() : 'Program Director';
        const evaluatingFacultyEmail = rowVals.length > COL_EMAIL_ADDRESS_FACULTY ? String(rowVals[COL_EMAIL_ADDRESS_FACULTY] || '').trim() : 'N/A';
        const positionOfFaculty = rowVals.length > COL_ASSESSOR_POSITION ? String(rowVals[COL_ASSESSOR_POSITION] || '').trim() : 'N/A';
        const residentName = studentObj && studentObj.name ? studentObj.name.trim() : 'Resident';
        const residentRollNumber = studentObj && studentObj.rollNumber ? String(studentObj.rollNumber).trim() : (rowVals.length > COL_RESIDENT_ROLL_NUMBER ? rowVals[COL_RESIDENT_ROLL_NUMBER] : 'N/A');
        const year = rowVals.length > COL_YEAR ? rowVals[COL_YEAR] : 'N/A';
        const specialty = rowVals.length > COL_SPECIALITY ? rowVals[COL_SPECIALITY] : 'N/A';
        const topic = rowVals.length > COL_TOPIC ? rowVals[COL_TOPIC] : 'N/A';
        const epaNumber = rowVals.length > COL_EPA_NUMBER ? rowVals[COL_EPA_NUMBER] : 'N/A';

        // Access individual rating scores using constants (0-based index)
        const historyRating = rowVals.length > COL_HISTORY ? rowVals[COL_HISTORY] : 'N/A';
        const clinicalRating = rowVals.length > COL_CLINICAL ? rowVals[COL_CLINICAL] : 'N/A';
        const investigationRating = rowVals.length > COL_INVESTIGATION ? rowVals[COL_INVESTIGATION] : 'N/A';
        const treatmentRating = rowVals.length > COL_TREATMENT ? rowVals[COL_TREATMENT] : 'N/A';
        const followupRating = rowVals.length > COL_FOLLOWUP ? rowVals[COL_FOLLOWUP] : 'N/A';
        const professionalismRating = rowVals.length > COL_PROFESSIONALISM ? rowVals[COL_PROFESSIONALISM] : 'N/A';
        const overallCareRating = rowVals.length > COL_OVERALL_CARE ? rowVals[COL_OVERALL_CARE] : 'N/A';

        // Get comments from the form response
        const areasDoneWell = rowVals.length > COL_AREAS_DONE_WELL ? (String(rowVals[COL_AREAS_DONE_WELL] || '')).trim() || 'Not specified' : 'Not specified';
        const actionNeeded = rowVals.length > COL_ACTION_NEEDED ? (String(rowVals[COL_ACTION_NEEDED] || '')).trim() || 'Not specified' : 'Not specified';


        // Access analytics data safely
        const analyticsData = analytics || { total: 0, pct: 0, discrimination: 'N/A', narrativeSkill: 'N/A' };
        const obtainedMarks = analyticsData.total || 0;
        const obtainedPercentage = (analyticsData.pct || 0).toFixed(2);
        const discriminationSkill = analyticsData.discrimination || 'N/A';
        const narrativeSkillAssessment = analyticsData.narrativeSkill || 'N/A';


        const pdEmail = pdObj && pdObj.email ? pdObj.email.trim() : '';
        const assessorEmail = rowVals.length > COL_EMAIL_ADDRESS_FACULTY ? String(rowVals[COL_EMAIL_ADDRESS_FACULTY] || '').trim() : ''; // Assessor email from form


        // Recipients for this email are Assessor and PD
        const recipients = [assessorEmail, pdEmail]
            .filter(email => email && email.trim() !== '')
            .filter((email, index, self) => self.indexOf(email) === index) // Remove duplicates
            .join(',');

        if (!recipients) {
            Logger.log('No valid recipient email addresses for Faculty/PD email. Skipping.');
            return;
        }

         // Safely get date string
        const dateValue = rowVals.length > COL_DATE ? rowVals[COL_DATE] : 'N/A';
        let dateStr = 'N/A';
        try {
            if (dateValue instanceof Date) {
                dateStr = dateValue.toLocaleDateString();
            } else {
                const parsedDate = new Date(dateValue);
                if (!isNaN(parsedDate.getTime())) {
                    dateStr = parsedDate.toLocaleDateString();
                } else {
                    dateStr = String(dateValue);
                }
            }
        } catch (e) { dateStr = String(dateValue); }


        // Construct the email body based on the user's draft structure, using CbD data
        const subject = `CbD Completed: ${residentName} (${specialty}) - ${dateStr}`; // Keep subject consistent

        const body = `Program Director: ${programDirectorName}\n` +
                     `Evaluating Faculty: ${evaluatingFacultyEmail}\n` + // Using email as name might not be available here
                     `Position of Faculty: ${positionOfFaculty}\n` +
                     `Resident Name: ${residentName}\n` +
                     `Resident Roll Number: ${residentRollNumber}\n` +
                     `Year: ${year}\n` +
                     `Speciality: ${specialty}\n` +
                     `EPA Topic: ${topic}\n` +
                     `EPA Number: ${epaNumber}\n\n` +

                     `Student Evaluation (Individual Ratings)\n` + // Adjusted header for CbD context
                     `History Taking: ${historyRating}\n` +
                     `Clinical Assessment: ${clinicalRating}\n` +
                     `Investigation and Referral: ${investigationRating}\n` +
                     `Treatment: ${treatmentRating}\n` +
                     `Follow-up and Future Planning: ${followupRating}\n` +
                     `Professionalism: ${professionalismRating}\n` +
                     `Overall Clinical Care: ${overallCareRating}\n\n` +
                     `Obtained Marks: ${obtainedMarks}\n` + // Include obtained marks and percentage
                     `Obtained Percentage: ${obtainedPercentage}%\n\n` +
                     `Areas done well: ${areasDoneWell}\n` + // Including comments from CbD form
                     `Specific action that needs to be taken to achieve desired competency: ${actionNeeded}\n\n` + // Including comments from CbD form

                     `Faculty Performance\n` +
                     `(This is based on scoring pattern and may not reflect truly, however it is helpful for program director to identify extreme pattern repeatedly.)\n\n` +

                     `The skill of faculty for identifying skills across various items seems to be ${discriminationSkill}.\n` +
                     `The skill of faculty in narrative writeup: ${narrativeSkillAssessment}.\n\n` +

                     `Thank you\n` + // Requested signature
                     `Dean Office`; // Requested signature


        MailApp.sendEmail({
            to: recipients,
            subject: subject,
            body: body
        });

        Logger.log('Email sent to Faculty/PD: ' + recipients);

    } catch (err) {
        Logger.log('Error sending email to Faculty/PD: ' + err.toString());
        if (err.stack) {
            Logger.log('Stack trace: ' + err.stack);
        }
        // Optionally send an error email to the script owner here
    }
}


/* === LOOKUP HELPERS (MODIFIED FOR ROLL NUMBER LOOKUP) =================== */

// This function now looks up a student by their Roll Number in the 'Residents' sheet.
// It expects the 'Residents' sheet to have: Full Name (Col A), Email (Col B), Roll Number (Col C).
function getStudentByRollNumber(rollNumber) {
    try {
        if (!rollNumber && String(rollNumber).trim() === '') { // Handle empty string specifically
            Logger.log('getStudentByRollNumber: No roll number provided or it was empty.');
            return null;
        }

        const ss = SpreadsheetApp.getActiveSpreadsheet();
        if (!ss) {
            Logger.log('getStudentByRollNumber: Could not access spreadsheet');
            // Optionally send an error email to the script owner here
            return null;
        }

        const sh = ss.getSheetByName('Residents');
        if (!sh) {
            Logger.log('Residents sheet not found. Cannot perform resident lookup.');
            // Optionally send an error email to the script owner here
            return null;
        }

        // Get data range safely
        let data;
        try {
            const dataRange = sh.getDataRange();
            if (!dataRange || dataRange.getValues().length <= 1) { // Check if sheet is empty or only has headers
                Logger.log('getStudentByRollNumber: Residents sheet is empty or only has headers');
                return null;
            }
            // Get all values from the data range. Ensure we get at least 3 columns for Name, Email, Roll Number.
            data = dataRange.getValues();

            Logger.log(`getStudentByRollNumber: Retrieved ${data.length} rows, ${data[0]?.length || 0} columns from Residents sheet.`);

        } catch (e) {
            Logger.log('getStudentByRollNumber: Error getting values: ' + e.toString());
            if (e.stack) {
                Logger.log('Stack trace: ' + e.stack);
            }
            // Optionally send an error email to the script owner here
            return null;
        }

        // Normalize the roll number for comparison (case-insensitive, no leading/trailing spaces)
        const normalizedRollNumber = String(rollNumber).trim().toLowerCase();
        Logger.log(`getStudentByRollNumber: Looking up student with normalized roll number: "${normalizedRollNumber}"`);

        // Debug: Log headers to confirm column mapping visually
        if (data.length > 0 && data[0].length >= 3) {
            Logger.log(`Residents Sheet Headers (first 3 columns): "${data[0][0]}", "${data[0][1]}", "${data[0][2]}"`);
            // Basic header validation check (optional but good practice)
            if (String(data[0][0]).trim().toLowerCase() !== 'full name' ||
                String(data[0][1]).trim().toLowerCase() !== 'email' ||
                String(data[0][2]).trim().toLowerCase() !== 'roll number') {
                Logger.log('WARNING: Residents sheet headers do not match expected format (Full Name | Email | Roll Number) in the first 3 columns.');
            }
        } else if (data.length > 0) {
            Logger.log(`Residents sheet has fewer than 3 columns in header row. Expected: Full Name | Email | Roll Number`);
        }


        // Loop through data starting from the second row (index 1) to find matching roll number
        for (let i = 1; i < data.length; i++) {
            // Skip rows with no data or not enough columns (need at least 3: Name, Email, Roll Number)
            // Check for data in Roll Number column (index 2 = Column C)
            if (!data[i] || data[i].length < 3 || String(data[i][2]).trim() === '') continue;

            const rowRollNumber = String(data[i][2]).trim().toLowerCase(); // Compare against Roll Number column (index 2)

            if (rowRollNumber === normalizedRollNumber) {
                Logger.log(`getStudentByRollNumber: Student found at row ${i + 1} by roll number "${rollNumber}": Name: "${data[i][0] || ''}", Email: "${data[i][1] || ''}"`);
                return {
                    name: data[i][0] || '',     // Get Name from Column A (index 0)
                    email: data[i][1] || '',    // Get Email from Column B (index 1)
                    rollNumber: data[i][2] || '' // Get Roll Number from Column C (index 2)
                };
            }
        }

        Logger.log(`getStudentByRollNumber: Student not found for roll number: "${rollNumber}"`);
        return null; // not found
    } catch (error) {
        Logger.log('Error in getStudentByRollNumber: ' + error.toString());
        if (error.stack) {
            Logger.log('Stack trace: ' + error.stack);
        }
        // Optionally send an error email to the script owner here
        return null;
    }
}

// This function remains unchanged, it looks up PD by Specialty in the 'ProgramDirector' sheet.
// It expects the 'ProgramDirector' sheet to have: Specialty (Col A), PD Name (Col B), Email (Col C).
function getProgramDirector(specialty) {
    try {
        if (!specialty && String(specialty).trim() === '') { // Handle empty string specifically
            Logger.log('getProgramDirector: No specialty provided or it was empty.');
            return null;
        }

        const ss = SpreadsheetApp.getActiveSpreadsheet();
        if (!ss) {
            Logger.log('getProgramDirector: Could not access spreadsheet');
             // Optionally send an error email to the script owner here
            return null;
        }

        const sh = ss.getSheetByName('ProgramDirector');
        if (!sh) {
            Logger.log('ProgramDirector sheet not found. Cannot perform PD lookup.');
            // Optionally send an error email to the script owner here
            return null;
        }

        // Get data range safely
        let data;
        try {
            const dataRange = sh.getDataRange();
            if (!dataRange || dataRange.getValues().length <= 1) { // Check if sheet is empty or only has headers
                Logger.log('getProgramDirector: ProgramDirector sheet is empty or only has headers');
                return null;
            }
            data = dataRange.getValues();

        } catch (e) {
            Logger.log('getProgramDirector: Error getting values: ' + e.toString());
            if (e.stack) {
                Logger.log('Stack trace: ' + e.stack);
            }
            // Optionally send an error email to the script owner here
            return null;
        }

        // Normalize the specialty for comparison (case-insensitive, no leading/trailing spaces)
        const normalizedSpecialty = String(specialty).trim().toLowerCase();
        Logger.log(`getProgramDirector: Looking up PD with normalized specialty: "${normalizedSpecialty}"`);

        // Debug: Log headers to confirm column mapping visually
        if (data.length > 0 && data[0].length >= 3) {
            Logger.log(`ProgramDirector Sheet Headers (first 3 columns): "${data[0][0]}", "${data[0][1]}", "${data[0][2]}"`);
            // Basic header validation check (optional but good practice)
             if (String(data[0][0]).trim().toLowerCase() !== 'specialty' ||
                 String(data[0][1]).trim().toLowerCase() !== 'pd name' ||
                 String(data[0][2]).trim().toLowerCase() !== 'email') {
                 Logger.log('WARNING: ProgramDirector sheet headers do not exactly match expected headers: ${expectedHeaders.join(' | ')}');
             }
        } else if (data.length > 0) {
            Logger.log(`ProgramDirector sheet has fewer than 3 columns in header row. Expected: Specialty | PD Name | Email`);
        }


        // Loop through data starting from the second row (index 1) to find matching program director
        for (let i = 1; i < data.length; i++) {
            // Skip rows with no data in first column (Specialty) or not enough columns
            // Check for data in Specialty column (index 0)
            if (!data[i] || data[i].length < 3 || String(data[i][0]).trim() === '') continue;

            const rowSpecialty = String(data[i][0]).trim().toLowerCase();
            if (rowSpecialty === normalizedSpecialty) {
                Logger.log(`getProgramDirector: PD found at row ${i + 1}: Specialty: "${data[i][0] || ''}", Name: "${data[i][1] || ''}", Email: "${data[i][2] || ''}"`);
                return {
                    specialty: data[i][0] || '', // Get Specialty from Column A (index 0)
                    name: data[i][1] || '',      // Get PD Name from Column B (index 1)
                    email: data[i][2] || ''      // Get Email from Column C (index 2)
                };
            }
        }

        Logger.log(`getProgramDirector: Program Director not found for specialty: "${specialty}"`);
        return null;
    } catch (error) {
        Logger.log('Error in getProgramDirector: ' + error.toString());
        if (error.stack) {
            Logger.log('Stack trace: ' + error.stack);
        }
        // Optionally send an error email to the script owner here
        return null;
    }
}

/* === SHEET BUILDER AND SETUP ==================================================== */
// This function sets up the CbD sheet with headers and basic formatting.
// It should be run manually once after linking a new form response sheet.
function setupCbDSheet() {
    try {
        const ss = SpreadsheetApp.getActiveSpreadsheet();
        if (!ss) {
            Logger.log('Could not access active spreadsheet');
            return null;
        }

        let sh = ss.getSheetByName('CbD');

        // Create new sheet if it doesn't exist
        if (!sh) {
            try {
                sh = ss.insertSheet('CbD', 0);
                Logger.log('Created new CbD sheet');
            } catch (err) {
                Logger.log('Error creating CbD sheet: ' + err.toString());
                // Optionally send an error email to the script owner here
                return null; // Stop if sheet creation fails
            }
        } else {
            Logger.log('CbD sheet already exists - ensuring headers and formatting are correct.');
             // Clear existing headers if sheet exists, to replace them accurately
             sh.getRange('1:1').clearContent();
        }

        // Define headers based *exactly* on the COLUMN MAP constants from Timestamp to Assessor Position
        // This list should match the order of the form response columns in your new sheet.
        const formResponseHeaders = [
            'Timestamp',
            'Email Address (Faculty)',
            'Resident Roll Number',
            'PG Program', // Matches COL_PG_PROGRAM
            'Year',       // Matches COL_YEAR
            'Date',
            'PG Program Specialty', // Matches COL_SPECIALITY (Note: Name differs slightly from constant name)
            'EPA Yes/No', // Matches COL_EPA_YES_NO
            'Topic',
            'EPA Number',
            'Case Description',
            'History Taking',        // Matches COL_HISTORY
            'Clinical Assessment',   // Matches COL_CLINICAL
            'Investigation and Referral', // Matches COL_INVESTIGATION
            'Treatment',             // Matches COL_TREATMENT
            'Follow-up and Future Planning', // Matches COL_FOLLOWUP
            'Professionalism',       // Matches COL_PROFESSIONALISM
            'Overall Clinical Care', // Matches COL_OVERALL_CARE
            'Areas done well',       // Matches COL_AREAS_DONE_WELL
            'Specific action needed',// Matches COL_ACTION_NEEDED
            'Assessing Doctor\'s Full Name', // Matches COL_ASSESSOR_NAME
            'Assessor\'s Position'   // Matches COL_ASSESSOR_POSITION
             // This list has 22 items, corresponding to indices 0-21 (Columns 1-22)
        ];

        // Define headers for the calculated analytics columns
         const analyticsHeaders = [
             'Calculated Full Mark',     // Matches COL_FULL_MARK (Index 23, Column 24)
             'Calculated Obtained Mark', // Matches COL_TOTAL_MARK (Index 24, Column 25) - Renamed for clarity
             'Percentage',               // Matches COL_PERCENTAGE (Index 25, Column 26)
             'Mean',                     // Matches COL_MEAN (Index 26, Column 27)
             'StanDev',                  // Matches COL_SD (Index 27, Column 28)
             'Variance',                 // Matches COL_VARIANCE (Index 28, Column 29)
             'Discrimination',           // Matches COL_DISCRIMINATION (Index 29, Column 30)
             'Narrative Skill'           // Matches COL_NARRATIVE_SKILL (Index 30, Column 31)
         ];


        try {
            // Set all headers: form response headers followed by analytics headers
            const allHeaders = formResponseHeaders.concat(analyticsHeaders);
            const fullHeaderRange = sh.getRange(1, 1, 1, allHeaders.length);
            fullHeaderRange.setValues([allHeaders]); // Set the new headers
            Logger.log('CbD sheet headers set/updated.');

            sh.setFrozenRows(1);
            fullHeaderRange.setFontWeight('bold').setWrap(true); // Apply formatting to the header range


            // Apply specific formatting to the analytics columns for data rows
            const totalRowsInSheet = sh.getMaxRows();
            const analyticsStartCol = COL_FULL_MARK + 1; // Analytics start column (1-based)

            // Apply formatting to existing data rows (if any)
             if (totalRowsInSheet > 1) {
                 // Apply general number format to the first 6 analytics columns
                 sh.getRange(2, analyticsStartCol, totalRowsInSheet - 1, 6).setNumberFormat('0.00');
                 // Apply percentage format specifically to the percentage column
                 sh.getRange(2, analyticsStartCol + 2, totalRowsInSheet - 1, 1).setNumberFormat('0.00%'); // Percentage is 3rd analytics column (index 2)
             }
             // Also apply format to a large range of future rows to ensure consistency
             const futureRowsToFormat = 1000; // Apply format to the next 1000 rows
             sh.getRange(sh.getLastRow() + 1, analyticsStartCol, futureRowsToFormat, 6).setNumberFormat('0.00');
             sh.getRange(sh.getLastRow() + 1, analyticsStartCol + 2, futureRowsToFormat, 1).setNumberFormat('0.00%'); // Percentage is 3rd analytics column (index 2)


        } catch (err) {
            Logger.log('Error setting up CbD sheet headers/formatting: ' + err.toString());
            if (err.stack) {
                 Logger.log('Stack trace: ' + err.stack);
            }
             // Optionally send an error email to the script owner here
            return null;
        }

        // --- Setup/Check Lookup Sheets ---
        // These functions check for the presence of lookup sheets but don't create them
        // You need to manually create and populate 'Residents' and 'ProgramDirector' sheets
         checkLookupSheetExists('Residents', ['Full Name', 'Email', 'Roll Number']);
         checkLookupSheetExists('ProgramDirector', ['Specialty', 'PD Name', 'Email']);


        Logger.log('setupCbDSheet completed.');

    } catch (error) {
        Logger.log('Error in setupCbDSheet: ' + error.toString());
        if (error.stack) {
            Logger.log('Stack trace: ' + error.stack);
        }
        // Optionally send an error email to the script owner here
    }
}

// Helper function to check if a lookup sheet exists and log a warning if not
function checkLookupSheetExists(sheetName, expectedHeaders) {
    const ss = SpreadsheetApp.getActiveSpreadsheet();
    const sh = ss.getSheetByName(sheetName);
    if (!sh) {
        Logger.log(`WARNING: Lookup sheet "${sheetName}" not found. Please create this sheet and populate it with data.`);
    } else {
        Logger.log(`Lookup sheet "${sheetName}" found.`);
        try {
            const headers = sh.getRange(1, 1, 1, expectedHeaders.length).getValues()[0];
            let headersMatch = true;
            for(let i = 0; i < expectedHeaders.length; i++) {
                if (String(headers[i]).trim().toLowerCase() !== String(expectedHeaders[i]).trim().toLowerCase()) {
                    headersMatch = false;
                    break;
                }
            }
             if (!headersMatch) {
                 Logger.log(`WARNING: Headers in "${sheetName}" sheet do not exactly match expected headers: ${expectedHeaders.join(' | ')}`);
             } else {
                  Logger.log(`Headers in "${sheetName}" sheet match expected format.`);
             }
        } catch(e) {
             Logger.log(`WARNING: Could not read headers from "${sheetName}" sheet to verify format.`);
        }
    }
}


/**
* CbD.gs – Case-based Discussion Automation (Add Individual Ratings to Resident Email 2025-05-17)
* -------------------------------------------------------------------------
* - Analytics start at column X (24) as per the constant map.
* - Uses Resident Roll Number from form (Col 3) to look up Name (Col 1) and Email (Col 2) in 'Residents' sheet (Roll Number expected in Col 3 of Residents).
* - Uses DOPS logic for Discrimination and Narrative Skill calculations.
* - Sends TWO separate emails: one detailed feedback to Resident, one notification to Faculty/Program Director.
* - Fixed TypeError when trimming numeric Roll Number.
* - Added "Thank you\nDean Office" at the end of both Resident and Faculty/PD emails.
* - Added individual rating scores to the Resident email body.
* - Added Assessor Name field from form response (used in Faculty/PD email).
* - MODIFIED: Defined missing COL_ASSESSOR_NAME constant.
* - MODIFIED: Adjusted COL_ASSESSOR_POSITION index to follow COL_ASSESSOR_NAME.
* - MODIFIED: Defined missing COL_FULL_MARK constant (used by writeAnalytics).
* - MODIFIED: Updated setupCbDSheet headers to match the column map constants.
* - MODIFIED: Changed email signature in Faculty/PD email to "Thank You\nDean Office".
* - MODIFIED: Updated the email body structure for Faculty/PD email based on user's provided draft, using available CbD data and individual ratings.
* - MODIFIED: Included resident details (Name, Roll Number, Year, Specialty, EPA Topic, EPA Number) and adjusted the structure in the email sent to the resident based on the user's latest draft.
* - **MODIFIED:** Included Assessor Email in the introductory sentence of the email sent to the resident, as Assessor Name from the form may not be available.
*
* HOW TO INSTALL
* 1. Ensure the linked Google Form writes responses in the exact order matching the COLUMN MAP below.
* 2. Run setupCbDSheet() once to build headers if sheet is empty and setup lookup sheets.
* Ensure 'Residents' sheet has 'Full Name' (Col A), 'Email' (Col B), and 'Roll Number' (Col C).
* Ensure 'ProgramDirector' sheet has 'Specialty' (Col A), 'PD Name' (Col B), and 'Email' (Col C).
* 3. Add an installable On-form-submit trigger → CbD_onSubmit. **Ensure this trigger is attached to the NEW linked Google Sheet.**
* 4. Maintain lookup sheets: "Residents" (Full Name | Email | Roll Number) and
* "ProgramDirector" (Specialty | PD Name | Email).
* -------------------------------------------------------------------------*/

/* === COLUMN MAP (1-based Sheet indices - converted to 0-based index for script access) ===============================
   Example: Column 1 is index 0, Column 21 is index 20.
   Form Response Columns (generally sequential from form)